In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from pytorch_tabnet.tab_model import TabNetClassifier
import torch


In [3]:
file_path = "data/SLIIT_IT_Student_Data_Template_filled-updated.csv"
df = pd.read_csv(file_path)

# Remove unwanted Excel "Unnamed" columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

target_col = "Experties_Preferred_Career_Field"
feature_cols = ["Soft_Skills", "Key_Skils", "Current_semester", "GPA"]

# Drop rows with missing target
df = df.dropna(subset=[target_col])

# Clean / type–fix feature columns
df["Soft_Skills"] = df["Soft_Skills"].fillna("")
df["Key_Skils"] = df["Key_Skils"].fillna("")

df["GPA"] = pd.to_numeric(df["GPA"], errors="coerce")
df = df.dropna(subset=["GPA"])

df["Current_semester"] = df["Current_semester"].astype(str)

print("Rows after cleaning:", len(df))
print(df[feature_cols + [target_col]].dtypes)


Rows after cleaning: 499
Soft_Skills                          object
Key_Skils                            object
Current_semester                     object
GPA                                 float64
Experties_Preferred_Career_Field     object
dtype: object


In [4]:
X = df[feature_cols].copy()
y = df[target_col].copy()

label_enc = LabelEncoder()
y_encoded = label_enc.fit_transform(y)

print("Class labels:")
for idx, cls in enumerate(label_enc.classes_):
    print(idx, "->", cls)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])


Class labels:
0 -> AI / Machine Learning Engineer
1 -> Cybersecurity / Network Engineer
2 -> Data Science & Analytics
3 -> DevOps / Cloud Engineer
4 -> Frontend / Web Developer
5 -> IT / Technology Generalist
6 -> Mobile App Developer
7 -> Software Engineer / Backend
8 -> UI/UX Designer
Train size: 399
Test size: 100


In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ("soft_tfidf", TfidfVectorizer(), "Soft_Skills"),
        ("key_tfidf", TfidfVectorizer(), "Key_Skils"),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Current_semester"]),
        ("num", StandardScaler(), ["GPA"])
    ]
)

# Fit preprocessor on training data
preprocessor.fit(X_train)

# Transform train and test
X_train_trans = preprocessor.transform(X_train)
X_test_trans = preprocessor.transform(X_test)

# Convert sparse -> dense and to float32 (TabNet expects numpy float32)
if not isinstance(X_train_trans, np.ndarray):
    X_train_dense = X_train_trans.toarray().astype(np.float32)
else:
    X_train_dense = X_train_trans.astype(np.float32)

if not isinstance(X_test_trans, np.ndarray):
    X_test_dense = X_test_trans.toarray().astype(np.float32)
else:
    X_test_dense = X_test_trans.astype(np.float32)

print("Transformed train shape:", X_train_dense.shape)
print("Transformed test shape:", X_test_dense.shape)

feature_names = preprocessor.get_feature_names_out()
print("Number of features:", len(feature_names))


Transformed train shape: (399, 729)
Transformed test shape: (100, 729)
Number of features: 729


In [7]:
n_features = X_train_dense.shape[1]
n_classes = len(label_enc.classes_)

tabnet_clf = TabNetClassifier(
    n_d=16,           # size of decision prediction layer
    n_a=16,           # size of attention layer
    n_steps=5,        # number of steps in TabNet
    gamma=1.5,        # relaxation parameter
    n_independent=2,
    n_shared=2,
    seed=42,
    verbose=1
)


D:\ResearchMy\Research_model\.venv\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [8]:
print("Training TabNet model...")

tabnet_clf.fit(
    X_train_dense, y_train,
    eval_set=[(X_train_dense, y_train), (X_test_dense, y_test)],
    eval_name=["train", "test"],
    eval_metric=["accuracy"],
    max_epochs=100,
    patience=20,
    batch_size=256,
    virtual_batch_size=128
)

print("✅ TabNet training completed.")


Training TabNet model...
epoch 0  | loss: 3.2064  | train_accuracy: 0.11529 | test_accuracy: 0.11    |  0:00:00s
epoch 1  | loss: 2.55887 | train_accuracy: 0.08772 | test_accuracy: 0.08    |  0:00:00s
epoch 2  | loss: 2.68047 | train_accuracy: 0.24561 | test_accuracy: 0.23    |  0:00:00s
epoch 3  | loss: 2.69109 | train_accuracy: 0.2406  | test_accuracy: 0.23    |  0:00:00s
epoch 4  | loss: 2.52068 | train_accuracy: 0.24561 | test_accuracy: 0.23    |  0:00:00s
epoch 5  | loss: 2.62936 | train_accuracy: 0.2406  | test_accuracy: 0.24    |  0:00:01s
epoch 6  | loss: 2.5994  | train_accuracy: 0.24561 | test_accuracy: 0.26    |  0:00:01s
epoch 7  | loss: 2.56558 | train_accuracy: 0.25564 | test_accuracy: 0.24    |  0:00:01s
epoch 8  | loss: 2.42496 | train_accuracy: 0.16792 | test_accuracy: 0.12    |  0:00:01s
epoch 9  | loss: 2.45693 | train_accuracy: 0.14536 | test_accuracy: 0.11    |  0:00:01s
epoch 10 | loss: 2.56878 | train_accuracy: 0.22306 | test_accuracy: 0.19    |  0:00:01s
epoch 1

D:\ResearchMy\Research_model\.venv\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [10]:
# Predict class indices
y_pred_tabnet = tabnet_clf.predict(X_test_dense)

tabnet_acc = accuracy_score(y_test, y_pred_tabnet)
print(f"\nTabNet Accuracy: {tabnet_acc:.4f}")

print("\nTabNet Classification Report:")
print(classification_report(y_test, y_pred_tabnet, target_names=label_enc.classes_))

print("\nTabNet Confusion Matrix (encoded labels):")
print(confusion_matrix(y_test, y_pred_tabnet))



TabNet Accuracy: 0.3000

TabNet Classification Report:
                                  precision    recall  f1-score   support

  AI / Machine Learning Engineer       0.00      0.00      0.00         7
Cybersecurity / Network Engineer       0.00      0.00      0.00         3
        Data Science & Analytics       0.32      0.93      0.48        27
         DevOps / Cloud Engineer       0.22      0.22      0.22        23
        Frontend / Web Developer       0.00      0.00      0.00        13
      IT / Technology Generalist       0.00      0.00      0.00         6
            Mobile App Developer       0.00      0.00      0.00         7
     Software Engineer / Backend       0.00      0.00      0.00         6
                  UI/UX Designer       0.00      0.00      0.00         8

                        accuracy                           0.30       100
                       macro avg       0.06      0.13      0.08       100
                    weighted avg       0.14      0.30 

D:\ResearchMy\Research_model\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\ResearchMy\Research_model\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\ResearchMy\Research_model\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0]

In [11]:
def predict_career_tabnet(soft_skills, key_skills, current_semester, gpa):
    """
    Predict career using trained TabNet + preprocessing + label encoder.
    """
    # 1. Build a one-row DataFrame
    new_df = pd.DataFrame([{
        "Soft_Skills": soft_skills,
        "Key_Skils": key_skills,
        "Current_semester": str(current_semester),
        "GPA": float(gpa)
    }])

    # 2. Preprocess using the fitted preprocessor
    X_new_trans = preprocessor.transform(new_df)
    if not isinstance(X_new_trans, np.ndarray):
        X_new_dense = X_new_trans.toarray().astype(np.float32)
    else:
        X_new_dense = X_new_trans.astype(np.float32)

    # 3. Predict class index with TabNet
    pred_encoded = tabnet_clf.predict(X_new_dense)[0]
    pred_label = label_enc.inverse_transform([pred_encoded])[0]

    # 4. Predict probabilities
    proba = tabnet_clf.predict_proba(X_new_dense)[0]  # probabilities for each class
    proba_dict = {
        label_enc.inverse_transform([i])[0]: float(p)
        for i, p in enumerate(proba)
    }

    return pred_label, proba_dict

# Example usage:
pred_career, proba_dict = predict_career_tabnet(
    soft_skills="Communication, Teamwork, Problem solving",
    key_skills="Python, SQL, Machine Learning",
    current_semester="2Y1S",
    gpa=3.4
)

print("\nPredicted career (TabNet):", pred_career)
print("\nClass probabilities:")
for cls, p in proba_dict.items():
    print(f"{cls:35s} -> {p:.3f}")



Predicted career (TabNet): Data Science & Analytics

Class probabilities:
AI / Machine Learning Engineer      -> 0.100
Cybersecurity / Network Engineer    -> 0.083
Data Science & Analytics            -> 0.151
DevOps / Cloud Engineer             -> 0.149
Frontend / Web Developer            -> 0.120
IT / Technology Generalist          -> 0.102
Mobile App Developer                -> 0.088
Software Engineer / Backend         -> 0.088
UI/UX Designer                      -> 0.119
